# Лабораторная работа №6: Построение и оценка VLM-пайплайна

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Запустить quantized VLM 2–4B.
- Создать evaluation JSONL.
- Сравнить prompts, resolution и decoding; разметить ошибки.


## Теоретическая часть

VLM связывает visual encoder, projector и language model. Оценка должна разделять captioning, VQA, OCR и structured extraction, фиксировать prompt, decoding и ошибки: hallucination, counting, spatial relations.

![Пример изображения для Visual Question Answering](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/vqa-example.png)


VLM (Vision-Language Model) — мультимодальная модель, которая работает одновременно с визуальной и текстовой информацией.

Из каких частей состоит модель?
1. Visual encoder — преобразует изображение в признаки.
2. Projector — преобразует визуальные признаки в формат, который может использовать языковая модель.
3. Language model — учитывает вопрос, визуальные признаки и контекст, после чего генерирует ответ.

**Квантование** позволяет запустить достаточно крупную нейросеть при ограниченной оперативной или видеопамяти. Оно позволяет хранить веса в более компактном представлении. Но при этом модель может стать менее точной

**Inference** — процесс, когда уже обученная модель получает входные данные и выдаёт предсказание.

**Decoding** — способ, которым модель выбирает следующий токен при генерации ответа.

 - *Greedy decoding* — жадное декодирование

    На каждом шаге выбирается токен с максимальной вероятностью:
    $$
    y_t = \arg\max_y P(y \mid y_{<t}, x)
    $$

    Здесь \(x\) — входное изображение и текст, а \(y_{<t}\) — уже сгенерированные токены.

 - *Sampling* — выборка из распределения

    Следующий токен выбирается случайно с учётом вероятностей модели. Температура изменяет форму распределения: при более низкой температуре вероятности концентрируются на наиболее вероятных токенах.

**Exact Match (EM)** — точное совпадение

Формула:
$$
EM_i =
\begin{cases}
1, & \hat y_i=y_i,\\
0, & \hat y_i\ne y_i.
\end{cases}
$$

Здесь:
- \(y_i\) — эталонный ответ;
- \(\hat y_i\) — предсказание модели.

Точность на наборе из \(N\) примеров:
$$
EM=\frac{1}{N}\sum_{i=1}^{N}EM_i.
$$

**ROUGE-L** — качество описания изображения

ROUGE-L оценивает сходство двух текстов через наибольшую общую подпоследовательность — LCS (Longest Common Subsequence).

**CIDEr** — оценка описаний с учётом информативности слов

CIDEr сравнивает сгенерированное описание с эталонными описаниями, используя взвешенные совпадения \(n\)-грамм — последовательностей из одного, двух и более слов. Часто встречающиеся общие слова получают меньший вес, а информативные совпадения — больший.

CER и WER — ошибки распознавания текста

Обе основаны на расстоянии редактирования Левенштейна — минимальном числе операций, необходимых, чтобы превратить один текст в другой.

Операции бывают трёх видов:
- Insertion (I) — вставка лишнего символа или слова;
- Deletion (D) — пропуск символа или слова;
- Substitution (S) — замена символа или слова другим.

Формулы:
$$
CER=\frac{S_c+D_c+I_c}{N_c},
$$

$$
WER=\frac{S_w+D_w+I_w}{N_w}.
$$

### Математическая постановка

Для открытых ответов одной exact match недостаточно. Используют нормализованный EM/F1, judge rubric и ручную разметку ошибок. Hallucination rate: $H=N_{unsupported}/N_{claims}$.


### Материалы

- [Transformers multimodal chat templates](https://huggingface.co/docs/transformers/chat_templating_multimodal)
- [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Запустить quantized VLM 2–4B.
2. Создать evaluation JSONL.
3. Сравнить prompts, resolution и decoding; разметить ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** VLM 4B NF4 inference, одно изображение.
- **Ограничение данных:** 20 evaluation examples: 5 на каждый из четырёх типов задач.
- **Fallback:** VLM 2B NF4 с тем же API.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и инференсу; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
DATA_ROOT = Path.cwd() / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'c:\\Anton\\16 триместр\\SOTA\\lab6\\data'}


In [2]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

In [3]:
# Библиотеки
from collections import Counter
import hashlib
import importlib

import open_eval_data

from collections import defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import textwrap

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

from datetime import datetime, timezone

import re
import string
import unicodedata

### Оценочный набор с происхождением

*Вес: 1 балл.*

**Постановка.** Используйте PixMo-Cap, PixMo-AskModelAnything, PixMo-Docs и PixMo-Count с опубликованными эталонами и сохраняйте происхождение каждой записи; локальных файлов от студента не требуется.

Выполните следующие действия:
1. Загрузите открытые изображения и опубликованные ответы через `open_eval_data.load_open_eval`; сохраните URL, split, индекс записи и сведения о лицензировании.
2. Проверьте размер набора и наличие captioning, VQA, OCR и structured extraction.

**Результат.** eval.jsonl с проверяемыми примерами и абсолютными либо существующими путями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
# FAST_DEV_RUN=True: 1 пример на задачу для smoke test.
# Полный запуск: 5 примеров на задачу, всего 20.
PER_TASK = 1 if FAST_DEV_RUN else 5
EXPECTED_TASKS = {"captioning", "vqa", "ocr", "structured"}

DATA_DIR = Path("data/lab06").resolve()
ARTIFACTS_DIR = Path("artifacts").resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = ARTIFACTS_DIR / "eval.jsonl"

# Загружаем реальные данные из опубликованных PixMo-наборов.
examples = open_eval_data.load_open_eval(
    root=str(DATA_DIR),
    per_task=PER_TASK,
)

# Проверяем состав и полноту набора.
counts = Counter(ex["task"] for ex in examples)

assert set(counts) == EXPECTED_TASKS, (
    f"Не все типы задач присутствуют: {dict(counts)}"
)
assert all(counts[task] == PER_TASK for task in EXPECTED_TASKS), (
    f"Неверное число примеров по задачам: {dict(counts)}"
)
assert len(examples) == PER_TASK * len(EXPECTED_TASKS)

required_fields = {
    "image", "question", "reference", "category",
    "task", "source", "source_split", "source_row",
    "image_url", "license",
}

# Подготавливаем JSONL и проверяем происхождение каждой записи.
records = []
seen_ids = set()

for ex in examples:
    record = dict(ex)

    image_path = Path(record["image"]).resolve()
    assert image_path.is_file(), f"Изображение не найдено: {image_path}"

    # Путь в артефакте всегда абсолютный.
    record["image"] = str(image_path)

    missing = required_fields - set(record)
    assert not missing, f"Отсутствуют поля: {missing}"

    assert record["source"], "Не указан источник датасета"
    assert record["source_split"], "Не указан split"
    assert record["source_row"] is not None, "Не указан индекс записи"
    assert record["license"], "Не указана информация о лицензии"

    # Создаём стабильный ID по источнику, split и индексу.
    record["id"] = (
        f'{record["task"]}:'
        f'{record["source_split"]}:'
        f'{record["source_row"]}'
    )
    assert record["id"] not in seen_ids, (
        f'Повторяющийся ID: {record["id"]}'
    )
    seen_ids.add(record["id"])

    # Контрольная сумма фактически сохранённого файла.
    if not record.get("image_sha256"):
        record["image_sha256"] = hashlib.sha256(
            image_path.read_bytes()
        ).hexdigest()

    # Явно указываем URL карточки исходного набора данных.
    # image_url отдельно хранит URL исходного изображения.
    record["source_dataset_url"] = record["source"]

    records.append(record)

# Сохраняем атомарно, чтобы при прерывании не оставить
# частично записанный eval.jsonl.
tmp_path = OUTPUT_PATH.with_suffix(".jsonl.tmp")

with tmp_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

tmp_path.replace(OUTPUT_PATH)

# Повторно читаем файл и проверяем сохранённый артефакт.
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    saved = [json.loads(line) for line in f if line.strip()]

assert len(saved) == len(records)
assert Counter(ex["task"] for ex in saved) == counts
assert all(Path(ex["image"]).is_absolute() for ex in saved)
assert all(ex["reference"] is not None for ex in saved)

print("Evaluation dataset создан")
print("Количество примеров:", len(saved))
print("Распределение задач:", dict(counts))
print("Файл:", OUTPUT_PATH)


Evaluation dataset создан
Количество примеров: 20
Распределение задач: {'captioning': 5, 'vqa': 5, 'ocr': 5, 'structured': 5}
Файл: C:\Anton\16 триместр\SOTA\lab6\artifacts\eval.jsonl


In [4]:
from IPython.display import display, Markdown

eval_path = Path("artifacts/eval.jsonl")

with eval_path.open("r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

task_names = {
    "captioning": "Captioning",
    "vqa": "Visual Question Answering",
    "ocr": "Document QA / OCR",
    "structured": "Structured Extraction"
}

rows = [
    "| Изображение | Тип задачи | Вопрос | Эталонный ответ | Источник | Split | Индекс | Лицензия |",
    "|---|---|---|---|---|---|---:|---|"
]

for ex in examples:
    image_path = Path(ex["image"]).resolve().as_posix()
    image_cell = f'<a href="file:///{image_path}">Открыть</a>'
    task = task_names.get(ex.get("task"), ex.get("task", ""))

    def clean(value):
        return (
            str(value if value is not None else "")
            .replace("|", "\\|")
            .replace("\n", " ")
            .replace("\r", " ")
        )

    rows.append(
        f"| {image_cell} | {clean(task)} | {clean(ex.get('question'))} "
        f"| {clean(ex.get('reference'))} | "
        f"[Ссылка]({clean(ex.get('source'))}) | "
        f"{clean(ex.get('source_split'))} | "
        f"{clean(ex.get('source_row'))} | "
        f"{clean(ex.get('license'))} |"
    )

display(Markdown("\n".join(rows)))

| Изображение | Тип задачи | Вопрос | Эталонный ответ | Источник | Split | Индекс | Лицензия |
|---|---|---|---|---|---|---:|---|
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-0.jpg">Открыть</a> | Captioning | Describe the image. | This photograph depicts a striking black bird, possibly a grackle or similar species, perched on a white cement wall with red stains. The bird's sleek, elongated body is adorned with iridescent feathers that shimmer with shades of blue, purple, and green, most prominently on its wings and back. Its piercing yellow eye and long, sharp beak lend it a fierce, almost aerodynamic appearance. The bird's dark, slender legs and short talons grip the edge of the cement structure, which resembles a divider or barrier, possibly in an outdoor setting like a park or building patio. The background is a blur of green and white hues, hinting at lush tropical plants and tall trees, setting a serene, natural scene. The bird is poised, looking towards the upper right-hand corner of the frame, with its long tail feathers trailing elegantly to the left. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-cap) | train | 0 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-2.jpg">Открыть</a> | Captioning | Describe the image. | The image showcases a dynamic, animated scene featuring four distinct characters, each housed within a large, square framing composed of a purple and lilac outer border.   1. The first character, located in the upper left corner, is an anthropomorphic creature with a purple body and a striking face that blends black and pink hues, complemented by blue and pink hair. This character stands on its hind limbs, wearing white pants and a matching white crop top that accentuates its animated, lively demeanor.  2. The second character, situated next to the first, exudes a monochromatic elegance. It is a black and white figure adorned in a sharp white coat, complemented by a black tie and a pristine white shirt. Its appearance is further distinguished by its white hair and piercing green eyes, contributing to its sophisticated and somewhat mysterious aura.  3. The third character, located in the lower left quadrant of the square, resembles a playful feline creature. It possesses four limbs with pink feet, transitioning to an orange hue from the knees upward. The creature boasts a whimsical tail that ends in purple fur, enhancing its quirky and endearing look. Its face, reminiscent of a cat, emphasizes its vibrant personality.  4. The final character, in the lower right quarter, features a white face contrasted by vibrant orange hair and accented with black horns. This character's distinctive appearance adds a touch of mystique to the ensemble.  At the bottom of the image, the text reads: "PayPal, coffee square, DM for all ad options, IOS," suggesting promotional content related to versatile payment and communication options. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-cap) | train | 2 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-3.jpg">Открыть</a> | Captioning | Describe the image. | A large brown suckerfish, commonly known as a plecostomus or pleco catfish, is prominently displayed inside a home fish tank. This species, recognized for its sucker-like mouth, clings to various surfaces within the tank, feeding on algae and other nutritional debris. This natural cleaning behavior helps maintain the tank's cleanliness, making it a popular choice among aquarium enthusiasts. In the image, the suckerfish is currently affixed to a piece of driftwood. Surrounding it are artificial plants anchored at the base of the aquarium, which is lined with black rocks. The plecostomus, often seen in household aquariums, contributes to the pristine condition of the tank by continuously scouring the glass and other surfaces, ensuring a clean and healthy environment. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-cap) | train | 3 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-7.jpg">Открыть</a> | Captioning | Describe the image. | In the Barrett-Jackson corporate office, an extraordinary conference room captures the essence of automotive passion and innovation. Dominating the room is a unique conference table, ingeniously incorporated around a vintage black 1970s Porsche 911. The classic sports car, sitting as the centerpiece, is encased in a large clear glass tabletop, transforming it into a functional yet remarkable discussion surface. Surrounding this one-of-a-kind table are several office chairs fashioned from racing seats, complete with roller wheels and armrests, epitomizing the fusion of comfort and high-octane style. The room's design boasts a color scheme of red, white, and black, with striking red lighting and accents complementing the white wall panels. The back wall prominently features the Barrett-Jackson logo, reiterating the prestigious automotive dealer's identity. Underneath the car, red neon lights cast a captivating glow, accentuating the vintage Porsche's sleek lines and reinforcing the room's thematic cohesion. This conference room is a testament to Barrett-Jackson's dedication to automotive excellence and distinctive style. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-cap) | train | 7 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-8.jpg">Открыть</a> | Captioning | Describe the image. | **Kane County COVID-19 Dashboard - Detailed Breakdown [Screenshot Capture: May 4, 2020, 9:22 PM]**  This image is a detailed snapshot of the COVID-19 tracking dashboard for Kane County, updated daily at 4:30 PM. As of the most recent update, the county reported a total of 2,154 confirmed COVID-19 cases, with 57 fatalities.   Key Features of the Dashboard:  1. **Date-Specific Data**:    - **Confirmed Cases by Date**: Illustrated through a line graph that charts the rising number of cases from April 1st to May 1st, highlighting an upward trajectory in infections over time.    2. **Geographical Distribution**:    - **Heat Map**: Using scatter plot visualization, the dashboard pinpoints infection hotspots across various geographical areas, including notable concentrations in Chicago, Kane County, and DuPage County.  3. **Demographic Breakdown**:    - **Gender Distribution**: Females show a higher infection rate compared to males.    - **Age Range Analysis**: The data is segmented into age groups (10-19, 30-39, 50-59, 70-79, 90+) to showcase age-specific infection rates.    - **Municipality Data**: The dashboard also offers a detailed breakdown of infection rates across different municipalities within the state.  This comprehensive snapshot offers valuable insights into the progression and distribution of COVID-19 within Kane County, facilitating informed decision-making and response strategies. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-cap) | train | 8 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-0.jpg">Открыть</a> | Visual Question Answering | What type of machine is this? | This is a vintage-style popcorn cart. It features a red body, glass front, and gold lettering. The cart is on wheels, making it mobile and perfect for events like carnivals, fairs, or movie theaters. It combines functionality with a charming 1950s aesthetic, ideal for making and displaying fresh popcorn. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-ask-model-anything) | train | 0 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-3.jpg">Открыть</a> | Visual Question Answering | What type of aircraft is in this image? | The aircraft in this image is a NASA Super Guppy. It's a specialized transport plane with a distinctive bulbous fuselage, used by NASA to carry oversized cargo like spacecraft components. The Super Guppy has a hinged nose for loading large items and four turboprop engines. Its unique design makes it one of the most recognizable aircraft in NASA's fleet. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-ask-model-anything) | train | 3 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-6.jpg">Открыть</a> | Visual Question Answering | how many vertical bars of rebar are there on the right side panel of this structure? | There are three vertical bars of rebar visible on the right side panel of this structure. These rebar rods can be clearly seen protruding from the top of the concrete block, providing reinforcement for the structure. The vertical orientation of these bars suggests they are integral to the structural integrity of the right side panel. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-ask-model-anything) | train | 6 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-7.jpg">Открыть</a> | Visual Question Answering | what is the frilly orange thing? | The frilly orange thing in the image is a feather duster worm. These fascinating marine creatures are a type of tube-dwelling polychaete worm. They get their name from their distinctive feather-like tentacles that extend from their tubular homes. These tentacles are usually brightly colored, often in shades of orange, pink, or yellow, and they fan out in a circular pattern, resembling a feather duster or a colorful flower.   Feather duster worms use their tentacles to filter feed, catching small particles and microorganisms from the water. When disturbed, they can quickly retract these tentacles into their protective tubes. These animals are commonly found in tropical and subtropical marine environments, often attached to coral reefs or rocky surfaces.  In underwater scenes, feather duster worms add a vibrant splash of color and movement, contributing to the rich biodiversity of coral reef ecosystems. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-ask-model-anything) | train | 7 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-8.jpg">Открыть</a> | Visual Question Answering | what kind of fish is the stripey little dude? | The stripey little fish in the image is a tiger barb. Tiger barbs are known for their distinctive striped pattern, which typically consists of several dark vertical stripes on a silver or golden body. These small, active fish are popular in aquariums due to their vibrant appearance and lively behavior. Tiger barbs are native to Southeast Asia and are members of the Cyprinidae family, which includes many other popular aquarium fish species. | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-ask-model-anything) | train | 8 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-0.jpg">Открыть</a> | Document QA / OCR | What is the total estimated cost? | $2,500 | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-docs) | validation | 0 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-1.jpg">Открыть</a> | Document QA / OCR | What are the strengths listed in the SWOT analysis? | Expertise in Niche Market, Community Trust | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-docs) | validation | 1 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-2.jpg">Открыть</a> | Document QA / OCR | what is Øystein's age? | 30 | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-docs) | validation | 2 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-3.jpg">Открыть</a> | Document QA / OCR | What is the key technology used for the artificial gravity system? | Centrifugal force-based system | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-docs) | validation | 3 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-4.jpg">Открыть</a> | Document QA / OCR | What is the name of the restaurant? | The Golden Nugget Grill | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-docs) | validation | 4 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-5.jpg">Открыть</a> | Structured Extraction | How many wild birds are visible? Return only JSON with keys label and count, using label 'wild birds'. | {"label": "wild birds", "count": 10} | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-count) | validation | 5 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-12.jpg">Открыть</a> | Structured Extraction | How many people are visible? Return only JSON with keys label and count, using label 'people'. | {"label": "people", "count": 9} | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-count) | validation | 12 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-15.jpg">Открыть</a> | Structured Extraction | How many airplanes are visible? Return only JSON with keys label and count, using label 'airplanes'. | {"label": "airplanes", "count": 6} | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-count) | validation | 15 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-17.jpg">Открыть</a> | Structured Extraction | How many people are visible? Return only JSON with keys label and count, using label 'people'. | {"label": "people", "count": 2} | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-count) | validation | 17 | ODC-BY-1.0 (dataset metadata; image rights may differ) |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-20.jpg">Открыть</a> | Structured Extraction | How many bottles are visible? Return only JSON with keys label and count, using label 'bottles'. | {"label": "bottles", "count": 7} | [Ссылка](https://huggingface.co/datasets/allenai/pixmo-count) | validation | 20 | ODC-BY-1.0 (dataset metadata; image rights may differ) |

### Модель и режим квантования

*Вес: 1 балл.*

**Постановка.** Фиксация версии модели и precision нужна для интерпретации результатов инференса.

Выполните следующие действия:
1. Загрузите processor и VLM.
2. На GPU включите NF4; без GPU не запускайте полный режим.

**Результат.** Объекты processor, model и model_id.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
# Для CPU начнём с модели 2B — она требует меньше памяти.
PRIMARY_MODEL_ID = os.getenv(
    "VLM_MODEL_ID", "Qwen/Qwen3-VL-2B-Instruct"
)
FALLBACK_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

model = None
processor = None
model_id = None
load_errors = []

for candidate_id in [PRIMARY_MODEL_ID, FALLBACK_MODEL_ID]:
    try:
        print(f"Загрузка модели на CPU: {candidate_id}")

        candidate_processor = AutoProcessor.from_pretrained(
            candidate_id
        )

        candidate_model = AutoModelForImageTextToText.from_pretrained(
            candidate_id,
            dtype=torch.float32,
            low_cpu_mem_usage=False,
        )
        candidate_model = candidate_model.to("cpu")
        candidate_model.eval()

        processor = candidate_processor
        model = candidate_model
        model_id = candidate_id
        break

    except Exception as exc:
        load_errors.append(f"{candidate_id}: {exc}")
        print(f"Не удалось загрузить {candidate_id}: {exc}")

        # Освобождаем ссылку на неудачно загруженную модель.
        candidate_model = None

if model is None:
    raise RuntimeError(
        "Не удалось загрузить ни одну модель. "
        "Проверьте версию Transformers и доступную RAM.\n"
        + "\n".join(load_errors)
    )

print("\nЗагрузка завершена.")
print("model_id:", model_id)
print("Processor:", type(processor).__name__)
print("Model:", type(model).__name__)
print("Устройство:", next(model.parameters()).device)
print("Квантование: отсутствует")
print("Режим: CPU / FP32")


Загрузка модели на CPU: Qwen/Qwen3-VL-2B-Instruct


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]


Загрузка завершена.
model_id: Qwen/Qwen3-VL-2B-Instruct
Processor: Qwen3VLProcessor
Model: Qwen3VLForConditionalGeneration
Устройство: cpu
Квантование: отсутствует
Режим: CPU / FP32


### Chat template и функция инференса

*Вес: 1 балл.*

**Постановка.** Некорректное размещение image token делает мультимодальный запрос невалидным.

Выполните следующие действия:
1. Соберите сообщение с изображением и вопросом.
2. Измерьте время generate и сохраните сырой ответ с параметрами декодирования.

**Результат.** Функции build_prompt и run_vlm.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [6]:
# Параметры декодирования по умолчанию.
GENERATION_PARAMS = {
    "max_new_tokens": 128,
    "do_sample": False,
}

ARTIFACTS_DIR = Path("artifacts").resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUTPUT_PATH = ARTIFACTS_DIR / "inference_raw.jsonl"


def build_prompt(image, question):
    """
    Формирует мультимодальное сообщение для chat template.
    image — путь к файлу или PIL.Image.Image.
    question — текст вопроса.
    """
    if isinstance(image, (str, Path)):
        image_path = Path(image).resolve()
        if not image_path.is_file():
            raise FileNotFoundError(f"Изображение не найдено: {image_path}")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("question должен быть непустой строкой")

    # Объект изображения передаём processor отдельно.
    # В сообщении сохраняем позицию image token.
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question.strip()},
            ],
        }
    ]

    return messages


def run_vlm(
    image,
    question,
    generation_params=None,
    save=True,
):
    """
    Запускает VLM, измеряет время генерации и при необходимости
    сохраняет сырой ответ вместе с параметрами декодирования.

    Возвращает словарь с ответом, временем и метаданными.
    """
    if model is None or processor is None or model_id is None:
        raise RuntimeError(
            "Сначала выполните ячейку загрузки модели и processor."
        )

    params = dict(GENERATION_PARAMS)
    if generation_params:
        params.update(generation_params)

    messages = build_prompt(image, question)

    # Открываем изображение и приводим его к RGB.
    if isinstance(image, (str, Path)):
        image_path = str(Path(image).resolve())
        with Image.open(image_path) as source:
            pil_image = source.convert("RGB")
    elif isinstance(image, Image.Image):
        pil_image = image.convert("RGB")
        image_path = None
    else:
        raise TypeError("image должен быть путём или объектом PIL.Image.Image")

    # Используем chat template модели — не вставляем image token вручную.
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[prompt_text],
        images=[pil_image],
        return_tensors="pt",
    )

    # Устройство должно совпадать с устройством модели.
    device = next(model.parameters()).device
    inputs = inputs.to(device)

    # Считаем время только генерации, без открытия файла и preprocessing.
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            **params,
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_seconds = time.perf_counter() - start_time

    # Убираем токены входного сообщения — декодируем только ответ модели.
    input_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, input_length:]

    raw_response = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    result = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_id": model_id,
        "image": image_path,
        "question": question,
        "raw_response": raw_response,
        "generation_params": params,
        "elapsed_seconds": round(elapsed_seconds, 4),
        "device": str(device),
        "precision": str(next(model.parameters()).dtype),
        "quantization": "none",
    }

    if save:
        with RAW_OUTPUT_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    return result


print("Функции build_prompt и run_vlm готовы.")


Функции build_prompt и run_vlm готовы.


### Возобновляемые предсказания

*Вес: 1 балл.*

**Постановка.** Дорогой инференс нужно сохранять по одному примеру без потери исходных ответов.

Выполните следующие действия:
1. Запустите VLM для каждого примера.
2. Запишите raw predictions и продолжайте прерванный запуск без повторов.

**Результат.** predictions.jsonl и predictions.json с фактическими выходами VLM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
EVAL_PATH = Path("artifacts/eval.jsonl").resolve()
PREDICTIONS_JSONL = Path("artifacts/predictions.jsonl").resolve()
PREDICTIONS_JSON = Path("artifacts/predictions.json").resolve()

PREDICTIONS_JSONL.parent.mkdir(parents=True, exist_ok=True)


def read_jsonl(path):
    """Читает JSONL, пропуская пустые строки."""
    if not path.exists():
        return []

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Повреждён JSONL {path}, строка {line_number}. "
                    "Исправьте эту строку перед продолжением."
                ) from exc
    return records


def save_predictions_json(records, path):
    """Атомарно обновляет JSON-файл со всеми сохранёнными предсказаниями."""
    tmp_path = path.with_suffix(".json.tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    tmp_path.replace(path)


# Загружаем evaluation-набор.
examples = read_jsonl(EVAL_PATH)
if not examples:
    raise RuntimeError(f"Оценочный набор пуст или отсутствует: {EVAL_PATH}")

# Каждой записи нужен стабильный ID для безопасного возобновления.
for index, ex in enumerate(examples):
    if not ex.get("id"):
        ex["id"] = (
            f'{ex.get("task", "unknown")}:'
            f'{ex.get("source_split", "unknown")}:'
            f'{ex.get("source_row", index)}'
        )

example_ids = [ex["id"] for ex in examples]
if len(example_ids) != len(set(example_ids)):
    raise ValueError("В eval.jsonl обнаружены повторяющиеся ID.")

# Загружаем checkpoint. Уже завершённые записи повторно не запускаем.
saved_records = read_jsonl(PREDICTIONS_JSONL)
completed = {}

for record in saved_records:
    record_id = record.get("id")
    if record_id:
        completed[record_id] = record

# Сохраняем порядок примеров исходного eval.jsonl.
pending = [ex for ex in examples if ex["id"] not in completed]

print(f"Всего примеров: {len(examples)}")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")

# Записываем результат каждого примера сразу после инференса.
with PREDICTIONS_JSONL.open("a", encoding="utf-8") as out:
    for index, ex in enumerate(pending, start=1):
        print(
            f'[{index}/{len(pending)}] '
            f'{ex["id"]} | {ex.get("task", "unknown")}'
        )

        result = run_vlm(
            image=ex["image"],
            question=ex["question"],
            save=False,  # Пишем в predictions.jsonl, не дублируем в другой лог.
        )

        record = {
            "id": ex["id"],
            "task": ex.get("task"),
            "category": ex.get("category"),
            "image": ex["image"],
            "question": ex["question"],
            "reference": ex.get("reference"),
            "source": ex.get("source"),
            "source_split": ex.get("source_split"),
            "source_row": ex.get("source_row"),
            "license": ex.get("license"),
            "model_id": result["model_id"],
            "raw_response": result["raw_response"],
            "generation_params": result["generation_params"],
            "elapsed_seconds": result["elapsed_seconds"],
            "device": result["device"],
            "precision": result["precision"],
            "quantization": result["quantization"],
            "timestamp_utc": result["timestamp_utc"],
        }

        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        out.flush()
        os.fsync(out.fileno())

        # Обновляем checkpoint в памяти и JSON-файл.
        completed[record["id"]] = record
        ordered_records = [
            completed[ex["id"]]
            for ex in examples
            if ex["id"] in completed
        ]
        save_predictions_json(ordered_records, PREDICTIONS_JSON)

        print(
            f'  Ответ: {record["raw_response"][:160]!r}'
            f' | Время: {record["elapsed_seconds"]:.2f} сек.'
        )

# Финальная проверка артефактов.
final_jsonl = read_jsonl(PREDICTIONS_JSONL)
final_by_id = {record.get("id"): record for record in final_jsonl}

expected_ids = set(example_ids)
missing_ids = expected_ids - set(final_by_id)

if missing_ids:
    print(f"Внимание: отсутствуют предсказания для {len(missing_ids)} примеров.")
else:
    final_ordered = [final_by_id[ex["id"]] for ex in examples]
    save_predictions_json(final_ordered, PREDICTIONS_JSON)
    print("\nВсе примеры обработаны.")

print("Строк в predictions.jsonl:", len(final_jsonl))
print("Покрытие текущего набора:", len(expected_ids & set(final_by_id)), "/", len(expected_ids))


Всего примеров: 20
FAST_DEV_RUN: False
[1/19] captioning:train:2 | captioning
  Ответ: 'This is a digital advertisement for four different character illustrations, presented in a grid format on a light blue background with a purple border. The imag' | Время: 119.53 сек.
[2/19] captioning:train:3 | captioning
  Ответ: 'This is a close-up photograph of a fish in an aquarium. The fish is the central subject, positioned on a piece of driftwood. It has a dark, almost black body, a' | Время: 1081.67 сек.
[3/19] captioning:train:7 | captioning
  Ответ: 'This is a high-end, modern conference room with a striking and unique design, centered around a custom-built table that is a replica of a Porsche 911. The room ' | Время: 80.13 сек.
[4/19] captioning:train:8 | captioning
  Ответ: 'This is a comprehensive dashboard from the Kane County Health Department, providing a visual summary of COVID-19 cases in Kane County, Illinois. The dashboard i' | Время: 66.54 сек.
[5/19] vqa:train:0 | vqa
  Ответ: 

In [7]:
pred_path = Path("artifacts/predictions.json")

with pred_path.open("r", encoding="utf-8") as f:
    predictions = json.load(f)

task_names = {
    "captioning": "Captioning",
    "vqa": "Visual Question Answering",
    "ocr": "Document QA / OCR",
    "structured": "Structured Extraction"
}

def clean(value):
    return (
        str(value if value is not None else "")
        .replace("|", "\\|")
        .replace("\n", " ")
        .replace("\r", " ")
    )

rows = [
    "| Изображение | Тип задачи | Вопрос | Эталонный ответ | Предсказание |",
    "|---|---|---|---|---|"
]

for p in predictions:
    image_path = Path(p["image"]).resolve().as_posix()
    image_cell = f'<a href="file:///{image_path}">Открыть</a>'
    task = task_names.get(p.get("task"), p.get("task", ""))

    rows.append(
        f'| {image_cell} | {clean(task)} | {clean(p.get("question"))} '
        f'| {clean(p.get("reference"))} | {clean(p.get("raw_response"))} |'
    )

display(Markdown("\n".join(rows)))

| Изображение | Тип задачи | Вопрос | Эталонный ответ | Предсказание |
|---|---|---|---|---|
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-0.jpg">Открыть</a> | Captioning | Describe the image. | This photograph depicts a striking black bird, possibly a grackle or similar species, perched on a white cement wall with red stains. The bird's sleek, elongated body is adorned with iridescent feathers that shimmer with shades of blue, purple, and green, most prominently on its wings and back. Its piercing yellow eye and long, sharp beak lend it a fierce, almost aerodynamic appearance. The bird's dark, slender legs and short talons grip the edge of the cement structure, which resembles a divider or barrier, possibly in an outdoor setting like a park or building patio. The background is a blur of green and white hues, hinting at lush tropical plants and tall trees, setting a serene, natural scene. The bird is poised, looking towards the upper right-hand corner of the frame, with its long tail feathers trailing elegantly to the left. | This is a close-up photograph of a single bird perched on a light blue, weathered concrete ledge. The bird is the central focus, captured in profile as it looks towards the right of the frame.  - **Bird's Appearance**: The bird is predominantly black, with a glossy, iridescent sheen on its back and wings that reflects a deep, rich blue-purple. It has a long, slender, pointed beak that is dark and slightly curved. Its eyes are a striking, bright yellow, and its legs and feet are a dark greyish-black, with sharp, curved claws gripping the ledge.  - **Posture |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-2.jpg">Открыть</a> | Captioning | Describe the image. | The image showcases a dynamic, animated scene featuring four distinct characters, each housed within a large, square framing composed of a purple and lilac outer border.   1. The first character, located in the upper left corner, is an anthropomorphic creature with a purple body and a striking face that blends black and pink hues, complemented by blue and pink hair. This character stands on its hind limbs, wearing white pants and a matching white crop top that accentuates its animated, lively demeanor.  2. The second character, situated next to the first, exudes a monochromatic elegance. It is a black and white figure adorned in a sharp white coat, complemented by a black tie and a pristine white shirt. Its appearance is further distinguished by its white hair and piercing green eyes, contributing to its sophisticated and somewhat mysterious aura.  3. The third character, located in the lower left quadrant of the square, resembles a playful feline creature. It possesses four limbs with pink feet, transitioning to an orange hue from the knees upward. The creature boasts a whimsical tail that ends in purple fur, enhancing its quirky and endearing look. Its face, reminiscent of a cat, emphasizes its vibrant personality.  4. The final character, in the lower right quarter, features a white face contrasted by vibrant orange hair and accented with black horns. This character's distinctive appearance adds a touch of mystique to the ensemble.  At the bottom of the image, the text reads: "PayPal, coffee square, DM for all ad options, IOS," suggesting promotional content related to versatile payment and communication options. | This is a digital advertisement for four different character illustrations, presented in a grid format on a light blue background with a purple border. The image is designed to be a "for sale" or "for hire" listing, with the artist's name, "SamS", visible on each artwork.  The four illustrations are arranged in a 2x2 grid:  - **Top-left:** A purple and white anthropomorphic character with a large, black, glowing mask. The character has a pink, spiky mane, a white and purple outfit, and a long, purple tail. The price is listed as $40. - ** |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-3.jpg">Открыть</a> | Captioning | Describe the image. | A large brown suckerfish, commonly known as a plecostomus or pleco catfish, is prominently displayed inside a home fish tank. This species, recognized for its sucker-like mouth, clings to various surfaces within the tank, feeding on algae and other nutritional debris. This natural cleaning behavior helps maintain the tank's cleanliness, making it a popular choice among aquarium enthusiasts. In the image, the suckerfish is currently affixed to a piece of driftwood. Surrounding it are artificial plants anchored at the base of the aquarium, which is lined with black rocks. The plecostomus, often seen in household aquariums, contributes to the pristine condition of the tank by continuously scouring the glass and other surfaces, ensuring a clean and healthy environment. | This is a close-up photograph of a fish in an aquarium. The fish is the central subject, positioned on a piece of driftwood. It has a dark, almost black body, and its head is covered in small, white, circular spots. The fish has a flat, wide mouth and a prominent, dark eye. Its fins are translucent and slightly translucent, with a subtle greenish tint. The fish is resting on a piece of driftwood that has a rough, textured surface with a mix of brown and tan colors. The background is filled with vibrant green artificial plants, which are arranged in a dense, lush manner. The water |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-7.jpg">Открыть</a> | Captioning | Describe the image. | In the Barrett-Jackson corporate office, an extraordinary conference room captures the essence of automotive passion and innovation. Dominating the room is a unique conference table, ingeniously incorporated around a vintage black 1970s Porsche 911. The classic sports car, sitting as the centerpiece, is encased in a large clear glass tabletop, transforming it into a functional yet remarkable discussion surface. Surrounding this one-of-a-kind table are several office chairs fashioned from racing seats, complete with roller wheels and armrests, epitomizing the fusion of comfort and high-octane style. The room's design boasts a color scheme of red, white, and black, with striking red lighting and accents complementing the white wall panels. The back wall prominently features the Barrett-Jackson logo, reiterating the prestigious automotive dealer's identity. Underneath the car, red neon lights cast a captivating glow, accentuating the vintage Porsche's sleek lines and reinforcing the room's thematic cohesion. This conference room is a testament to Barrett-Jackson's dedication to automotive excellence and distinctive style. | This is a high-end, modern conference room with a striking and unique design, centered around a custom-built table that is a replica of a Porsche 911. The room is designed to be a high-performance, luxury environment, likely for a company or individual with a strong passion for automotive excellence.  ### Key Features of the Room  - **The Central Table**: The most prominent feature is a large, rectangular table made of clear glass. The glass is not just a surface; it is a direct replica of the front of a black Porsche 911, including the iconic front grille, headlights, and the distinctive red accent stripe |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-cap-default-train-8.jpg">Открыть</a> | Captioning | Describe the image. | **Kane County COVID-19 Dashboard - Detailed Breakdown [Screenshot Capture: May 4, 2020, 9:22 PM]**  This image is a detailed snapshot of the COVID-19 tracking dashboard for Kane County, updated daily at 4:30 PM. As of the most recent update, the county reported a total of 2,154 confirmed COVID-19 cases, with 57 fatalities.   Key Features of the Dashboard:  1. **Date-Specific Data**:    - **Confirmed Cases by Date**: Illustrated through a line graph that charts the rising number of cases from April 1st to May 1st, highlighting an upward trajectory in infections over time.    2. **Geographical Distribution**:    - **Heat Map**: Using scatter plot visualization, the dashboard pinpoints infection hotspots across various geographical areas, including notable concentrations in Chicago, Kane County, and DuPage County.  3. **Demographic Breakdown**:    - **Gender Distribution**: Females show a higher infection rate compared to males.    - **Age Range Analysis**: The data is segmented into age groups (10-19, 30-39, 50-59, 70-79, 90+) to showcase age-specific infection rates.    - **Municipality Data**: The dashboard also offers a detailed breakdown of infection rates across different municipalities within the state.  This comprehensive snapshot offers valuable insights into the progression and distribution of COVID-19 within Kane County, facilitating informed decision-making and response strategies. | This is a comprehensive dashboard from the Kane County Health Department, providing a visual summary of COVID-19 cases in Kane County, Illinois. The dashboard is updated daily at 4:30 PM and is designed to offer a clear, data-driven overview of the pandemic's status.  The dashboard is divided into several sections, each presenting different aspects of the data:  - **Top Section: Summary Statistics**   - The main headline reads "Kane County COVID-19 Dashboard".   - It shows a total of **2,154 confirmed COVID-19 cases**, with **57 deaths**.   - The data |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-0.jpg">Открыть</a> | Visual Question Answering | What type of machine is this? | This is a vintage-style popcorn cart. It features a red body, glass front, and gold lettering. The cart is on wheels, making it mobile and perfect for events like carnivals, fairs, or movie theaters. It combines functionality with a charming 1950s aesthetic, ideal for making and displaying fresh popcorn. | This is a popcorn machine, specifically a popcorn cart. It's a portable, commercial-grade machine used to make popcorn, often found in movie theaters, food stands, or at events. The cart design allows it to be easily moved and is typically used for serving popcorn to customers. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-3.jpg">Открыть</a> | Visual Question Answering | What type of aircraft is in this image? | The aircraft in this image is a NASA Super Guppy. It's a specialized transport plane with a distinctive bulbous fuselage, used by NASA to carry oversized cargo like spacecraft components. The Super Guppy has a hinged nose for loading large items and four turboprop engines. Its unique design makes it one of the most recognizable aircraft in NASA's fleet. | The aircraft in the image is the NASA C-17 Globemaster III, a large, four-engine, high-wing, transport aircraft. It is a modified version of the C-130 Hercules, designed for long-range transport and cargo missions. The C-17 is known for its ability to carry a wide range of payloads, including troops, vehicles, and supplies, and is used by the U.S. Air Force and other military branches. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-6.jpg">Открыть</a> | Visual Question Answering | how many vertical bars of rebar are there on the right side panel of this structure? | There are three vertical bars of rebar visible on the right side panel of this structure. These rebar rods can be clearly seen protruding from the top of the concrete block, providing reinforcement for the structure. The vertical orientation of these bars suggests they are integral to the structural integrity of the right side panel. | Based on the provided image, we can analyze the structure to determine the number of vertical rebar bars on the right side panel.  - The image shows a concrete column that is in the process of being constructed. - The rebar (reinforcing steel) is wrapped around the column, forming a cage-like structure. - The rebar is arranged in a grid pattern, with vertical bars running up and down the column. - The right side panel of the structure is the one on the right edge of the image, which is the side of the rebar cage.  Looking closely at the rebar cage on the right side of the column |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-7.jpg">Открыть</a> | Visual Question Answering | what is the frilly orange thing? | The frilly orange thing in the image is a feather duster worm. These fascinating marine creatures are a type of tube-dwelling polychaete worm. They get their name from their distinctive feather-like tentacles that extend from their tubular homes. These tentacles are usually brightly colored, often in shades of orange, pink, or yellow, and they fan out in a circular pattern, resembling a feather duster or a colorful flower.   Feather duster worms use their tentacles to filter feed, catching small particles and microorganisms from the water. When disturbed, they can quickly retract these tentacles into their protective tubes. These animals are commonly found in tropical and subtropical marine environments, often attached to coral reefs or rocky surfaces.  In underwater scenes, feather duster worms add a vibrant splash of color and movement, contributing to the rich biodiversity of coral reef ecosystems. | Based on the image provided, the frilly orange thing is a type of marine worm known as a **sea anemone** or more specifically, a **sea fan worm**.  However, the most accurate identification is that it is a **sea anemone**. The image shows a close-up of a sea anemone, which is a type of marine invertebrate that lives in the ocean. The frilly, orange, and white tentacles are the oral arms of the sea anemone, which are used for capturing food and for defense. The central part of the anemone is the mouth, which is |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-ask-model-anything-default-train-8.jpg">Открыть</a> | Visual Question Answering | what kind of fish is the stripey little dude? | The stripey little fish in the image is a tiger barb. Tiger barbs are known for their distinctive striped pattern, which typically consists of several dark vertical stripes on a silver or golden body. These small, active fish are popular in aquariums due to their vibrant appearance and lively behavior. Tiger barbs are native to Southeast Asia and are members of the Cyprinidae family, which includes many other popular aquarium fish species. | Based on the visual characteristics in the image, the "stripey little dude" is a **tiger barb**.  Here's a breakdown of the identification:  -   **Coloration:** The fish has a distinct pattern of bold, black vertical stripes on a yellowish or white background. This is the classic "tiger" pattern. -   **Body Shape:** It has a slender, elongated body, which is typical for this species. -   **Size:** The fish appears to be a small to medium-sized aquarium fish. -   **Common Name:** This specific pattern is known as the "tiger barb" |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-0.jpg">Открыть</a> | Document QA / OCR | What is the total estimated cost? | $2,500 | Based on the provided report, the total estimated cost is $2,500. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-1.jpg">Открыть</a> | Document QA / OCR | What are the strengths listed in the SWOT analysis? | Expertise in Niche Market, Community Trust | Based on the SWOT analysis provided in the image, the strengths are:  - Expertise in Niche Market: Specialized knowledge in stock car tuning attracts racing enthusiasts. - Community Trust: Strong local relationships foster customer loyalty and word-of-mouth promotion. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-2.jpg">Открыть</a> | Document QA / OCR | what is Øystein's age? | 30 | Based on the "Personal Details" section of the Preconception Health Record, Øystein's age is 30. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-3.jpg">Открыть</a> | Document QA / OCR | What is the key technology used for the artificial gravity system? | Centrifugal force-based system | The key technology used for the artificial gravity system is a centrifugal force-based system. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-docs-other-validation-4.jpg">Открыть</a> | Document QA / OCR | What is the name of the restaurant? | The Golden Nugget Grill | Based on the menu provided, the name of the restaurant is **The Golden Nugget Grill**. This is displayed prominently at the top of the menu. |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-5.jpg">Открыть</a> | Structured Extraction | How many wild birds are visible? Return only JSON with keys label and count, using label 'wild birds'. | {"label": "wild birds", "count": 10} | ```json {   "label": "wild birds",   "count": 10 } ``` |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-12.jpg">Открыть</a> | Structured Extraction | How many people are visible? Return only JSON with keys label and count, using label 'people'. | {"label": "people", "count": 9} | ```json {   "label": "people",   "count": 9 } ``` |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-15.jpg">Открыть</a> | Structured Extraction | How many airplanes are visible? Return only JSON with keys label and count, using label 'airplanes'. | {"label": "airplanes", "count": 6} | ```json {   "label": "airplanes",   "count": 5 } ``` |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-17.jpg">Открыть</a> | Structured Extraction | How many people are visible? Return only JSON with keys label and count, using label 'people'. | {"label": "people", "count": 2} | ```json {   "label": "people",   "count": 2 } ``` |
| <a href="file:///C:/Anton/16 триместр/SOTA/lab6/data/lab06/pixmo-count-default-validation-20.jpg">Открыть</a> | Structured Extraction | How many bottles are visible? Return only JSON with keys label and count, using label 'bottles'. | {"label": "bottles", "count": 7} | ```json {   "label": "bottles",   "count": 7 } ``` |

### Точная оценка коротких ответов

*Вес: 1 балл.*

**Постановка.** Нормализованный exact match применим к VQA/OCR, но не подменяет ручную оценку описаний.

Выполните следующие действия:
1. Нормализуйте предсказания и эталоны для VQA/OCR.
2. Измерьте точность по категориям.

**Результат.** metrics.json с реально измеренным exact match.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
PREDICTIONS_PATH = Path("artifacts/predictions.json")
METRICS_PATH = Path("artifacts/metrics.json")
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)


def normalize_answer(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()

    # Удаляем пунктуацию, включая Unicode-пунктуацию.
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )

    # Стандартная нормализация для англоязычных QA-ответов.
    text = re.sub(r"\b(a|an|the)\b", " ", text)

    # Приводим последовательности пробелов к одному пробелу.
    return " ".join(text.split())


if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл предсказаний: {PREDICTIONS_PATH}. "
        "Сначала выполните ячейку возобновляемого инференса."
    )

with PREDICTIONS_PATH.open("r", encoding="utf-8") as f:
    predictions = json.load(f)

if not predictions:
    raise ValueError("Файл predictions.json не содержит предсказаний.")

# EM считаем только для VQA и OCR.
eligible_tasks = {"vqa", "ocr"}
evaluated = []

for record in predictions:
    if record.get("task") not in eligible_tasks:
        continue

    reference = record.get("reference")
    prediction = record.get("raw_response")

    if reference is None or prediction is None:
        continue

    normalized_reference = normalize_answer(reference)
    normalized_prediction = normalize_answer(prediction)

    evaluated.append({
        "id": record.get("id"),
        "task": record["task"],
        "category": record.get("category", record["task"]),
        "reference": reference,
        "prediction": prediction,
        "normalized_reference": normalized_reference,
        "normalized_prediction": normalized_prediction,
        "exact_match": normalized_reference == normalized_prediction,
    })

if not evaluated:
    raise ValueError(
        "Нет готовых предсказаний для VQA/OCR. "
        "Проверьте task, reference и raw_response в predictions.json."
    )

# Группируем результаты отдельно по задаче и категории.
groups = defaultdict(list)

for item in evaluated:
    groups[("task", item["task"])].append(item)
    groups[("category", item["category"])].append(item)

metrics = {
    "metric": "normalized_exact_match",
    "normalization": [
        "Unicode NFKC",
        "lowercase",
        "remove punctuation",
        "remove English articles a/an/the",
        "collapse whitespace",
    ],
    "evaluated_tasks": sorted(eligible_tasks),
    "excluded_tasks": ["captioning", "structured"],
    "total_evaluated": len(evaluated),
    "correct": sum(item["exact_match"] for item in evaluated),
    "exact_match": sum(item["exact_match"] for item in evaluated) / len(evaluated),
    "groups": {},
    "per_example": evaluated,
}

for (group_type, group_name), items in sorted(groups.items()):
    correct = sum(item["exact_match"] for item in items)
    metrics["groups"].setdefault(group_type, {})[group_name] = {
        "count": len(items),
        "correct": correct,
        "exact_match": correct / len(items),
    }

# Атомарная запись метрик.
tmp_path = METRICS_PATH.with_suffix(".json.tmp")
with tmp_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
    f.flush()

tmp_path.replace(METRICS_PATH)

# Человекочитаемый вывод.
print(f"Всего оценено ответов VQA/OCR: {metrics['total_evaluated']}")
print(f"Правильных ответов: {metrics['correct']}")
print(f"Общий normalized EM: {metrics['exact_match']:.3f}")

print("\nПо задачам:")
for task, result in metrics["groups"]["task"].items():
    print(
        f"  {task}: EM={result['exact_match']:.3f} "
        f"({result['correct']}/{result['count']})"
    )

print("\nПо категориям:")
for category, result in metrics["groups"]["category"].items():
    print(
        f"  {category}: EM={result['exact_match']:.3f} "
        f"({result['correct']}/{result['count']})"
    )

print("\nМетрики сохранены:", METRICS_PATH.resolve())


Всего оценено ответов VQA/OCR: 10
Правильных ответов: 0
Общий normalized EM: 0.000

По задачам:
  ocr: EM=0.000 (0/5)
  vqa: EM=0.000 (0/5)

По категориям:
  document_qa: EM=0.000 (0/5)
  open_vqa: EM=0.000 (0/5)

Метрики сохранены: C:\Anton\16 триместр\SOTA\lab6\artifacts\metrics.json


In [9]:
PREDICTIONS_PATH = Path("artifacts/predictions.json")
EXTRA_METRICS_PATH = Path("artifacts/extra_metrics.json")


def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    return " ".join(text.split())


def tokenize(text):
    return re.findall(r"\w+", normalize_text(text))


def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, 1):
        current = [i]

        for j, char_b in enumerate(b, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (char_a != char_b)
            ))

        previous = current

    return previous[-1]


def parse_json_answer(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        result = json.loads(text)
        return result if isinstance(result, dict) else None
    except (json.JSONDecodeError, TypeError):
        return None


def rouge_l(reference, prediction):
    ref = tokenize(reference)
    pred = tokenize(prediction)

    if not ref or not pred:
        return 0.0

    # Длина наибольшей общей подпоследовательности.
    previous = [0] * (len(pred) + 1)

    for a in ref:
        current = [0]

        for j, b in enumerate(pred, 1):
            current.append(
                previous[j - 1] + 1 if a == b
                else max(previous[j], current[-1])
            )

        previous = current

    lcs = previous[-1]
    precision = lcs / len(pred)
    recall = lcs / len(ref)

    return (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )


with PREDICTIONS_PATH.open("r", encoding="utf-8") as f:
    predictions = json.load(f)

results = {}

# 1. Captioning: ROUGE-L и дополнительные CIDEr/SPICE при наличии библиотеки.
captions = [
    p for p in predictions
    if p.get("task") == "captioning"
    and p.get("reference") is not None
    and p.get("raw_response") is not None
]

if captions:
    scores = [
        rouge_l(p["reference"], p["raw_response"])
        for p in captions
    ]

    results["captioning"] = {
        "metric": "ROUGE-L",
        "count": len(scores),
        "mean_rouge_l": sum(scores) / len(scores),
    }

    # CIDEr и SPICE требуют pycocoevalcap.
    try:
        from pycocoevalcap.cider.cider import Cider

        references = {
            i: [str(p["reference"])]
            for i, p in enumerate(captions)
        }
        hypotheses = {
            i: [str(p["raw_response"])]
            for i, p in enumerate(captions)
        }

        cider_score, _ = Cider().compute_score(references, hypotheses)

        results["captioning"]["CIDEr"] = float(cider_score)

    except ImportError:
        results["captioning"]["CIDEr/SPICE"] = (
            "Не рассчитаны: установите pycocoevalcap"
        )
    except Exception as e:
        import traceback
        traceback.print_exc()
        results["captioning"]["CIDEr/SPICE"] = (
            f"Не рассчитаны: {type(e).__name__}: {e}"
        )


# 2. OCR: Character Error Rate (CER) и Word Error Rate (WER).
ocr = [
    p for p in predictions
    if p.get("task") == "ocr"
    and p.get("reference") is not None
    and p.get("raw_response") is not None
]

if ocr:
    char_errors = char_total = 0
    word_errors = word_total = 0

    for p in ocr:
        ref_chars = normalize_text(p["reference"])
        pred_chars = normalize_text(p["raw_response"])

        char_errors += edit_distance(ref_chars, pred_chars)
        char_total += len(ref_chars)

        ref_words = tokenize(p["reference"])
        pred_words = tokenize(p["raw_response"])

        word_errors += edit_distance(ref_words, pred_words)
        word_total += len(ref_words)

    results["ocr"] = {
        "count": len(ocr),
        "CER": char_errors / char_total if char_total else 0.0,
        "WER": word_errors / word_total if word_total else 0.0,
    }


# 3. Structured extraction: точность полей и micro-F1.
structured = [
    p for p in predictions
    if p.get("task") == "structured"
    and p.get("reference") is not None
    and p.get("raw_response") is not None
]

if structured:
    correct_fields = total_fields = 0
    tp = fp = fn = 0
    valid_pairs = 0

    for p in structured:
        ref = parse_json_answer(p["reference"])
        pred = parse_json_answer(p["raw_response"])

        if ref is None or pred is None:
            continue

        valid_pairs += 1
        keys = set(ref) | set(pred)

        for key in keys:
            ref_has = key in ref
            pred_has = key in pred

            if ref_has:
                total_fields += 1

            correct = (
                ref_has and pred_has
                and normalize_text(str(ref[key]))
                == normalize_text(str(pred[key]))
            )

            if correct:
                correct_fields += 1
                tp += 1
            else:
                if pred_has:
                    fp += 1
                if ref_has:
                    fn += 1

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    results["structured"] = {
        "count": len(structured),
        "valid_json_pairs": valid_pairs,
        "field_accuracy": (
            correct_fields / total_fields if total_fields else 0.0
        ),
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        ),
    }


# Сохраняем результаты.
with EXTRA_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Выводим метрики.
for task, values in results.items():
    print(f"\n{task.upper()}")

    for metric, value in values.items():
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")
        else:
            print(f"  {metric}: {value}")

print(f"\nДополнительные метрики сохранены: {EXTRA_METRICS_PATH.resolve()}")


CAPTIONING
  metric: ROUGE-L
  count: 5
  mean_rouge_l: 0.1903
  CIDEr: 0.0088

OCR
  count: 5
  CER: 5.4369
  WER: 5.2353

STRUCTURED
  count: 5
  valid_json_pairs: 5
  field_accuracy: 0.9000
  micro_precision: 0.9000
  micro_recall: 0.9000
  micro_f1: 0.9000

Дополнительные метрики сохранены: C:\Anton\16 триместр\SOTA\lab6\artifacts\extra_metrics.json


### Заготовка экспертной разметки

*Вес: 1 балл.*

**Постановка.** Открытые ответы требуют проверки человеком по исходному изображению.

Выполните следующие действия:
1. Сформируйте отдельные поля correctness и типологии ошибок.
2. Не перезаписывайте ранее заполненную разметку.

**Результат.** manual_review.jsonl с исходными ответами и полями экспертной оценки.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
PREDICTIONS_PATH = Path("artifacts/predictions.json")
MANUAL_REVIEW_PATH = Path("artifacts/manual_review.jsonl")

# Поля экспертной разметки
REVIEW_FIELDS = {
    "correctness": None,
    "error_types": [],
    "review_comment": "",
}

# Допустимые значения для последующей ручной разметки.
# correctness: correct / partially_correct / incorrect / unverifiable
# error_types: список типов ошибок из предложенной типологии.
ERROR_TYPES = [
    "object_or_entity",
    "attribute",
    "counting",
    "text_or_ocr",
    "spatial_relation",
    "missing_information",
    "hallucination",
    "instruction_following",
    "other",
]

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл предсказаний: {PREDICTIONS_PATH.resolve()}"
    )

with PREDICTIONS_PATH.open("r", encoding="utf-8") as f:
    predictions = json.load(f)

# Загружаем существующую разметку, если файл уже есть.
existing_reviews = {}

if MANUAL_REVIEW_PATH.exists():
    with MANUAL_REVIEW_PATH.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            record_id = record.get("id")

            if record_id is None:
                raise ValueError(
                    f"В строке {line_number} нет поля 'id'."
                )

            if record_id in existing_reviews:
                raise ValueError(
                    f"Дублирующийся id в разметке: {record_id}"
                )

            existing_reviews[record_id] = record

# Создаём запись для каждого предсказания.
# Ранее размеченные поля не перезаписываем.
merged_reviews = []

for prediction in predictions:
    record_id = prediction.get("id")

    if record_id is None:
        raise ValueError("У предсказания отсутствует поле 'id'.")

    if record_id in existing_reviews:
        review = existing_reviews[record_id].copy()

        # Добавляем новые исходные поля, только если их ещё нет.
        for key, value in prediction.items():
            review.setdefault(key, value)

        # Добавляем поля разметки только при их отсутствии.
        for key, value in REVIEW_FIELDS.items():
            review.setdefault(key, value)
    else:
        review = prediction.copy()
        review.update(REVIEW_FIELDS)

    merged_reviews.append(review)

# Атомарная запись JSONL.
MANUAL_REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
tmp_path = MANUAL_REVIEW_PATH.with_suffix(".jsonl.tmp")

with tmp_path.open("w", encoding="utf-8", newline="\n") as f:
    for review in merged_reviews:
        f.write(json.dumps(review, ensure_ascii=False) + "\n")
    f.flush()

tmp_path.replace(MANUAL_REVIEW_PATH)

print(f"Предсказаний: {len(predictions)}")
print(f"Записей в разметке: {len(merged_reviews)}")
print(f"Файл разметки: {MANUAL_REVIEW_PATH.resolve()}")
print(f"Типы ошибок: {ERROR_TYPES}")


Предсказаний: 20
Записей в разметке: 20
Файл разметки: C:\Anton\16 триместр\SOTA\lab6\artifacts\manual_review.jsonl
Типы ошибок: ['object_or_entity', 'attribute', 'counting', 'text_or_ocr', 'spatial_relation', 'missing_information', 'hallucination', 'instruction_following', 'other']


### Ручная типология ошибок

*Вес: 1 балл.*

**Постановка.** Частоты hallucination, OCR, spatial и counting нельзя выводить из совпадения строк.

Выполните следующие действия:
1. Заполните ручные поля по изображению и эталону.
2. Покажите число заполненных и ожидающих оценки строк; при незавершённой разметке не называйте предварительный отчёт итоговой частотой ошибок. Форма создаётся внутри ноутбука.

**Результат.** error_report.json с числом реально просмотренных примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
review_path = Path("artifacts/manual_review.jsonl")
report_path = Path("artifacts/error_report.json")

with review_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

reviewed = [
    r for r in records
    if r.get("correctness") is not None
    and isinstance(r.get("error_types"), list)
]

pending = len(records) - len(reviewed)

counts = Counter(
    error
    for r in reviewed
    for error in set(r["error_types"])
)

report = {
    "total_examples": len(records),
    "reviewed_examples": len(reviewed),
    "pending_examples": pending,
    "error_counts": dict(counts),
    "error_rates": {
        error: round(count / len(reviewed), 4)
        for error, count in counts.items()
    } if reviewed else {},
    "is_final": pending == 0,
}

report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(report, ensure_ascii=False, indent=2))
print(f"\nСохранено: {report_path.resolve()}")

{
  "total_examples": 20,
  "reviewed_examples": 20,
  "pending_examples": 0,
  "error_counts": {
    "missing_information": 3,
    "attribute": 1,
    "object_or_entity": 2,
    "counting": 1
  },
  "error_rates": {
    "missing_information": 0.15,
    "attribute": 0.05,
    "object_or_entity": 0.1,
    "counting": 0.05
  },
  "is_final": true
}

Сохранено: C:\Anton\16 триместр\SOTA\lab6\artifacts\error_report.json


### Парная абляция prompt и разрешения

*Вес: 1 балл.*

**Постановка.** Варианты модели сравниваются только на одном наборе изображений.

Выполните следующие действия:
1. Переберите шаблоны, разрешения и режимы декодирования.
2. Инкрементально сохраните реальные ответы для каждой конфигурации.

**Результат.** ablation.jsonl с предсказаниями VLM для каждого варианта.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
ARTIFACTS_DIR = Path("artifacts")
EVAL_PATH = ARTIFACTS_DIR / "eval.jsonl"
ABLATION_PATH = ARTIFACTS_DIR / "ablation.jsonl"

# FAST_DEV_RUN использует тот же небольшой smoke-test набор.
with EVAL_PATH.open("r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f if line.strip()]

# Ограничиваем варианты для практичного запуска на CPU.
PROMPT_TEMPLATES = {
    "direct": "{question}",
    "concise": (
        "Answer the following question accurately and concisely. "
        "Do not add unnecessary explanation.\nQuestion: {question}"
    ),
}

RESOLUTIONS = {
    "original": None,
    "512": (512, 512),
}

DECODING_MODES = {
    "greedy": {"do_sample": False, "max_new_tokens": 128},
    "sampling": {
        "do_sample": True,
        "temperature": 0.7,
        "max_new_tokens": 128,
    },
}

# Индексируем уже сохранённые результаты, чтобы можно было продолжить запуск.
completed = set()
if ABLATION_PATH.exists():
    with ABLATION_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed.add((
                    item["id"],
                    item["prompt_template"],
                    item["resolution"],
                    item["decoding_mode"],
                ))

total = (
    len(eval_records)
    * len(PROMPT_TEMPLATES)
    * len(RESOLUTIONS)
    * len(DECODING_MODES)
)
print(f"Примеров: {len(eval_records)}; всего конфигураций: {total}")

with ABLATION_PATH.open("a", encoding="utf-8") as out:
    for record in eval_records:
        for prompt_name, prompt_template in PROMPT_TEMPLATES.items():
            for resolution_name, target_size in RESOLUTIONS.items():
                for decoding_name, generation_params in DECODING_MODES.items():
                    key = (
                        record["id"],
                        prompt_name,
                        resolution_name,
                        decoding_name,
                    )
                    if key in completed:
                        continue

                    image_path = Path(record["image"])
                    if not image_path.is_file():
                        raise FileNotFoundError(image_path)

                    with Image.open(image_path) as source_image:
                        image = source_image.convert("RGB")

                    if target_size is not None:
                        image.thumbnail(target_size, Image.Resampling.LANCZOS)

                    question = prompt_template.format(
                        question=record["question"]
                    )

                    start = time.perf_counter()
                    result = run_vlm(
                        image,
                        question,
                        generation_params=generation_params,
                        save=False,
                    )
                    elapsed = time.perf_counter() - start

                    item = {
                        "id": record["id"],
                        "task": record.get("task"),
                        "category": record.get("category"),
                        "image": record["image"],
                        "question": record["question"],
                        "reference": record.get("reference"),
                        "prompt_template": prompt_name,
                        "resolution": resolution_name,
                        "decoding_mode": decoding_name,
                        "generation_params": generation_params,
                        "raw_response": result["raw_response"],
                        "elapsed_seconds": round(elapsed, 4),
                        "model_id": result["model_id"],
                        "device": result["device"],
                        "precision": result["precision"],
                        "quantization": result["quantization"],
                    }

                    out.write(json.dumps(item, ensure_ascii=False) + "\n")
                    out.flush()
                    completed.add(key)

                    print(
                        f"[{len(completed)}/{total}] {record['id']} | "
                        f"{prompt_name} | {resolution_name} | {decoding_name}"
                    )

print(f"\nРезультаты сохранены в: {ABLATION_PATH.resolve()}")


Примеров: 20; всего конфигураций: 160
[90/160] ocr:validation:1 | direct | original | sampling
[91/160] ocr:validation:1 | direct | 512 | greedy
[92/160] ocr:validation:1 | direct | 512 | sampling
[93/160] ocr:validation:1 | concise | original | greedy
[94/160] ocr:validation:1 | concise | original | sampling
[95/160] ocr:validation:1 | concise | 512 | greedy
[96/160] ocr:validation:1 | concise | 512 | sampling
[97/160] ocr:validation:2 | direct | original | greedy
[98/160] ocr:validation:2 | direct | original | sampling
[99/160] ocr:validation:2 | direct | 512 | greedy
[100/160] ocr:validation:2 | direct | 512 | sampling
[101/160] ocr:validation:2 | concise | original | greedy
[102/160] ocr:validation:2 | concise | original | sampling
[103/160] ocr:validation:2 | concise | 512 | greedy
[104/160] ocr:validation:2 | concise | 512 | sampling
[105/160] ocr:validation:3 | direct | original | greedy
[106/160] ocr:validation:3 | direct | original | sampling
[107/160] ocr:validation:3 | direc

### Качество абляционных вариантов

*Вес: 1 балл.*

**Постановка.** Изменение prompt или разрешения следует сопоставлять по измеримому качеству коротких ответов.

Выполните следующие действия:
1. Для VQA/OCR рассчитайте нормализованный exact match по каждому варианту.
2. Сохраните размер и точность каждой группы.

**Результат.** ablation_quality.json с фактической точностью VQA/OCR.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
ABLATION_PATH = Path("artifacts/ablation.jsonl")
OUTPUT_PATH = Path("artifacts/ablation_quality.json")

def normalize_answer(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = "".join(
        " " if unicodedata.category(ch).startswith("P") else ch
        for ch in text
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

with ABLATION_PATH.open("r", encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f if line.strip()]

groups = defaultdict(list)

for record in predictions:
    if record.get("task") not in {"vqa", "ocr"}:
        continue

    reference = record.get("reference")
    prediction = record.get("raw_response")

    if reference is None or prediction is None:
        continue

    key = (
        record["task"],
        record["prompt_template"],
        record["resolution"],
        record["decoding_mode"],
    )

    groups[key].append(
        normalize_answer(reference) == normalize_answer(prediction)
    )

results = []

for (task, prompt, resolution, decoding), matches in sorted(groups.items()):
    results.append({
        "task": task,
        "prompt_template": prompt,
        "resolution": resolution,
        "decoding_mode": decoding,
        "sample_size": len(matches),
        "correct": sum(matches),
        "normalized_exact_match": round(sum(matches) / len(matches), 4),
    })

report = {
    "metric": "normalized_exact_match",
    "normalization": (
        "Unicode NFKC, lowercase, punctuation-to-spaces, "
        "remove articles a/an/the, collapse whitespace"
    ),
    "tasks": ["vqa", "ocr"],
    "total_groups": len(results),
    "results": results,
    "note": (
        "FAST_DEV_RUN results are smoke-test results, "
        "not final quality estimates."
    ),
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Групп: {len(results)}")
for item in results:
    print(
        f"{item['task']:4} | prompt={item['prompt_template']:8} | "
        f"resolution={item['resolution']:8} | "
        f"decoding={item['decoding_mode']:8} | "
        f"n={item['sample_size']:2} | "
        f"EM={item['normalized_exact_match']:.3f}"
    )

print(f"\nОтчёт сохранён: {OUTPUT_PATH.resolve()}")


Групп: 16
ocr  | prompt=concise  | resolution=512      | decoding=greedy   | n= 5 | EM=0.800
ocr  | prompt=concise  | resolution=512      | decoding=sampling | n= 5 | EM=0.800
ocr  | prompt=concise  | resolution=original | decoding=greedy   | n= 5 | EM=0.800
ocr  | prompt=concise  | resolution=original | decoding=sampling | n= 5 | EM=0.800
ocr  | prompt=direct   | resolution=512      | decoding=greedy   | n= 5 | EM=0.000
ocr  | prompt=direct   | resolution=512      | decoding=sampling | n= 5 | EM=0.000
ocr  | prompt=direct   | resolution=original | decoding=greedy   | n= 5 | EM=0.000
ocr  | prompt=direct   | resolution=original | decoding=sampling | n= 5 | EM=0.000
vqa  | prompt=concise  | resolution=512      | decoding=greedy   | n= 5 | EM=0.000
vqa  | prompt=concise  | resolution=512      | decoding=sampling | n= 5 | EM=0.000
vqa  | prompt=concise  | resolution=original | decoding=greedy   | n= 5 | EM=0.000
vqa  | prompt=concise  | resolution=original | decoding=sampling | n= 5 | EM=

In [16]:
from itertools import product

ABLATION_PATH = Path("artifacts/ablation.jsonl")
ABLATION_METRICS_PATH = Path("artifacts/ablation_metrics.json")

PROMPT_TEMPLATES = ["direct", "concise"]
RESOLUTIONS = ["original", "512"]
DECODING_MODES = ["greedy", "sampling"]


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


def calculate_metrics(records):
    results = {}

    # 1. Captioning: ROUGE-L и CIDEr
    captions = [
        p for p in records
        if p.get("task") == "captioning"
        and p.get("reference") is not None
        and p.get("raw_response") is not None
    ]

    if captions:
        scores = [
            rouge_l(p["reference"], p["raw_response"])
            for p in captions
        ]

        results["captioning"] = {
            "metric": "ROUGE-L",
            "count": len(scores),
            "mean_rouge_l": sum(scores) / len(scores),
        }

        try:
            from pycocoevalcap.cider.cider import Cider

            references = {
                i: [str(p["reference"])]
                for i, p in enumerate(captions)
            }
            hypotheses = {
                i: [str(p["raw_response"])]
                for i, p in enumerate(captions)
            }

            cider_score, _ = Cider().compute_score(
                references, hypotheses
            )

            results["captioning"]["CIDEr"] = float(cider_score)

        except ImportError:
            results["captioning"]["CIDEr/SPICE"] = (
                "Не рассчитаны: установите pycocoevalcap"
            )
        except Exception as e:
            results["captioning"]["CIDEr/SPICE"] = (
                f"Не рассчитаны: {type(e).__name__}: {e}"
            )

    # 2. OCR: CER и WER
    ocr = [
        p for p in records
        if p.get("task") == "ocr"
        and p.get("reference") is not None
        and p.get("raw_response") is not None
    ]

    if ocr:
        char_errors = char_total = 0
        word_errors = word_total = 0

        for p in ocr:
            ref_chars = normalize_text(p["reference"])
            pred_chars = normalize_text(p["raw_response"])

            char_errors += edit_distance(ref_chars, pred_chars)
            char_total += len(ref_chars)

            ref_words = tokenize(p["reference"])
            pred_words = tokenize(p["raw_response"])

            word_errors += edit_distance(ref_words, pred_words)
            word_total += len(ref_words)

        results["ocr"] = {
            "count": len(ocr),
            "CER": (
                char_errors / char_total if char_total else 0.0
            ),
            "WER": (
                word_errors / word_total if word_total else 0.0
            ),
        }

    # 3. Structured extraction: field accuracy и micro-F1
    structured = [
        p for p in records
        if p.get("task") == "structured"
        and p.get("reference") is not None
        and p.get("raw_response") is not None
    ]

    if structured:
        correct_fields = total_fields = 0
        tp = fp = fn = 0
        valid_pairs = 0

        for p in structured:
            ref = parse_json_answer(p["reference"])
            pred = parse_json_answer(p["raw_response"])

            if ref is None or pred is None:
                continue

            valid_pairs += 1
            keys = set(ref) | set(pred)

            for key in keys:
                ref_has = key in ref
                pred_has = key in pred

                if ref_has:
                    total_fields += 1

                correct = (
                    ref_has and pred_has
                    and normalize_text(str(ref[key]))
                    == normalize_text(str(pred[key]))
                )

                if correct:
                    correct_fields += 1
                    tp += 1
                else:
                    if pred_has:
                        fp += 1
                    if ref_has:
                        fn += 1

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0

        results["structured"] = {
            "count": len(structured),
            "valid_json_pairs": valid_pairs,
            "field_accuracy": (
                correct_fields / total_fields
                if total_fields else 0.0
            ),
            "micro_precision": precision,
            "micro_recall": recall,
            "micro_f1": (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            ),
        }

    return results


# Загружаем все эксперименты из JSONL.
predictions = load_jsonl(ABLATION_PATH)

# Формируем 8 комбинаций: 2 * 2 * 2.
all_results = {}

for prompt, resolution, decoding in product(
    PROMPT_TEMPLATES,
    RESOLUTIONS,
    DECODING_MODES,
):
    block_name = (
        f"prompt={prompt}"
        f"__resolution={resolution}"
        f"__decoding={decoding}"
    )

    # Строго фильтруем записи по всем трём факторам.
    subset = [
        p for p in predictions
        if p.get("prompt_template") == prompt
        and str(p.get("resolution")) == resolution
        and p.get("decoding_mode") == decoding
    ]

    metrics = calculate_metrics(subset)

    all_results[block_name] = {
        "configuration": {
            "prompt_template": prompt,
            "resolution": resolution,
            "decoding_mode": decoding,
        },
        "total_records": len(subset),
        "metrics": metrics,
    }

# Сохраняем все 8 блоков, включая пустые комбинации.
ABLATION_METRICS_PATH.parent.mkdir(
    parents=True, exist_ok=True
)

with ABLATION_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# Выводим метрики каждого блока.
for block_name, block in all_results.items():
    print(f"\n{'=' * 70}")
    print(block_name)
    print(f"Всего записей: {block['total_records']}")

    if not block["metrics"]:
        print("Метрики не рассчитаны: нет подходящих записей.")
        continue

    for task, values in block["metrics"].items():
        print(f"\n{task.upper()}")

        for metric, value in values.items():
            if isinstance(value, float):
                print(f"  {metric}: {value:.4f}")
            else:
                print(f"  {metric}: {value}")

print(
    f"\nРезультаты сохранены: "
    f"{ABLATION_METRICS_PATH.resolve()}"
)


prompt=direct__resolution=original__decoding=greedy
Всего записей: 20

CAPTIONING
  metric: ROUGE-L
  count: 5
  mean_rouge_l: 0.1903
  CIDEr: 0.0088

OCR
  count: 5
  CER: 5.4369
  WER: 5.2353

STRUCTURED
  count: 5
  valid_json_pairs: 5
  field_accuracy: 0.9000
  micro_precision: 0.9000
  micro_recall: 0.9000
  micro_f1: 0.9000

prompt=direct__resolution=original__decoding=sampling
Всего записей: 20

CAPTIONING
  metric: ROUGE-L
  count: 5
  mean_rouge_l: 0.2074
  CIDEr: 0.0123

OCR
  count: 5
  CER: 8.3981
  WER: 7.5294

STRUCTURED
  count: 5
  valid_json_pairs: 5
  field_accuracy: 0.9000
  micro_precision: 0.9000
  micro_recall: 0.9000
  micro_f1: 0.9000

prompt=direct__resolution=512__decoding=greedy
Всего записей: 20

CAPTIONING
  metric: ROUGE-L
  count: 5
  mean_rouge_l: 0.2046
  CIDEr: 0.0038

OCR
  count: 5
  CER: 7.5437
  WER: 7.4706

STRUCTURED
  count: 5
  valid_json_pairs: 5
  field_accuracy: 0.9000
  micro_precision: 0.9000
  micro_recall: 0.9000
  micro_f1: 0.9000

pro

## Вывод

Для описания изображений лучше всего подробное описание (промпт), не сжатое изображение, более мягкое распределение токенов

Для задачи OCR лучше всего краткий ответ, не сжатое изображение

Для STRUCTURED не удалось выяснить лучшую конфигурацию

### Задержка и сводная оценка

*Вес: 1 балл.*

**Постановка.** Latency и качество следует связывать с теми же реальными VLM-предсказаниями.

Выполните следующие действия:
1. Вычислите median и p95 задержки каждого варианта.
2. Сохраните сводку качества, ручных ошибок и бюджета.

**Результат.** eval_summary.json с фактическими метриками и числом изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [17]:
import math
import statistics
from collections import defaultdict

ARTIFACTS = Path("artifacts")

def read_jsonl(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def read_json(path):
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def percentile(values, p):
    if not values:
        return None
    values = sorted(values)
    position = (len(values) - 1) * p
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return round(values[lower], 4)
    return round(
        values[lower] * (upper - position)
        + values[upper] * (position - lower),
        4,
    )

# Реальные ответы, полученные в абляционном эксперименте
ablation = read_jsonl(ARTIFACTS / "ablation.jsonl")
quality = read_json(ARTIFACTS / "ablation_quality.json")
manual = read_jsonl(ARTIFACTS / "manual_review.jsonl")
error_report = read_json(ARTIFACTS / "error_report.json")
predictions = read_json(ARTIFACTS / "predictions.json")

# Задержка для каждой конфигурации абляции
latency_groups = defaultdict(list)

for r in ablation:
    latency = r.get("elapsed_seconds")
    if latency is None:
        continue

    key = (
        r.get("prompt_template"),
        r.get("resolution"),
        r.get("decoding_mode"),
    )
    latency_groups[key].append(float(latency))

latency_summary = []
for (prompt, resolution, decoding), values in sorted(latency_groups.items()):
    latency_summary.append({
        "prompt_template": prompt,
        "resolution": resolution,
        "decoding_mode": decoding,
        "n_predictions": len(values),
        "median_seconds": round(statistics.median(values), 4),
        "p95_seconds": percentile(values, 0.95),
    })

# Сводка качества из рассчитанных метрик
quality_summary = quality.get("results", [])

# Ручные ошибки — используем уже созданный отчёт.
# Если его нет, считаем по заполненным ручным полям.
if not error_report and manual:
    reviewed = [
        r for r in manual
        if r.get("correctness") is not None
        and isinstance(r.get("error_types"), list)
    ]
    counts = Counter(
        error
        for r in reviewed
        for error in set(r["error_types"])
    )
    error_summary = {
        "total_examples": len(manual),
        "reviewed_examples": len(reviewed),
        "pending_examples": len(manual) - len(reviewed),
        "error_counts": dict(counts),
        "is_final": len(reviewed) == len(manual),
    }
else:
    error_summary = error_report

# Бюджет: число изображений и реально сохранённых предсказаний.
image_paths = {
    r.get("image")
    for r in predictions
    if r.get("image")
}
image_paths.update(
    r.get("image")
    for r in ablation
    if r.get("image")
)

summary = {
    "num_images": len(image_paths),
    "num_base_predictions": len(predictions),
    "num_ablation_predictions": len(ablation),
    "latency_by_configuration": latency_summary,
    "quality": quality_summary,
    "manual_error_summary": error_summary,
    "budget": {
        "model_id": (
            predictions[0].get("model_id")
            if predictions else (
                ablation[0].get("model_id") if ablation else None
            )
        ),
        "device": (
            predictions[0].get("device")
            if predictions else (
                ablation[0].get("device") if ablation else None
            )
        ),
        "precision": (
            predictions[0].get("precision")
            if predictions else (
                ablation[0].get("precision") if ablation else None
            )
        ),
        "quantization": (
            predictions[0].get("quantization")
            if predictions else (
                ablation[0].get("quantization") if ablation else None
            )
        ),
        "total_inference_seconds": round(
            sum(float(r.get("elapsed_seconds", 0))
                for r in ablation),
            4,
        ),
    },
    "fast_dev_run": None,
    "note": (
        "Сводка рассчитана по сохранённым предсказаниям. "
        "Latency включает только измеренные инференсы из ablation.jsonl. "
        "Проверь FAST_DEV_RUN в настройках ноутбука перед сдачей."
    ),
}

# По возможности читаем флаг FAST_DEV_RUN из глобального состояния ноутбука.
if "FAST_DEV_RUN" in globals():
    summary["fast_dev_run"] = bool(FAST_DEV_RUN)

output_path = ARTIFACTS / "eval_summary.json"
output_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Изображений: {summary['num_images']}")
print(f"Базовых предсказаний: {summary['num_base_predictions']}")
print(f"Абляционных предсказаний: {summary['num_ablation_predictions']}")
print(f"Конфигураций с задержкой: {len(latency_summary)}")
print(f"FAST_DEV_RUN: {summary['fast_dev_run']}")
print(f"Сводка сохранена: {output_path.resolve()}")


Изображений: 20
Базовых предсказаний: 20
Абляционных предсказаний: 160
Конфигураций с задержкой: 8
FAST_DEV_RUN: False
Сводка сохранена: C:\Anton\16 триместр\SOTA\lab6\artifacts\eval_summary.json


## Выводы

При сжатии изображения и генерации краткого ответа медианное время ответа уменьшается в среднем в 4 раза.

Короткий ответ для задачи OCR сильно улучшает оценку. 


### Творческий бонус: проверка формулировки вопроса

*Вес: 1 дополнительный балл.*

**Постановка.** Уточнение вопроса может изменить ответ модели без смены изображения.

Выполните следующие действия:
1. Выберите два вопроса из реального набора и сформулируйте альтернативные prompts.
2. Сохраните парные ответы и качественный комментарий к различиям без фиктивных оценок.

**Результат.** bonus_prompts.json с исходными и новыми ответами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [18]:
ARTIFACTS = Path("artifacts")
eval_path = ARTIFACTS / "eval.jsonl"
output_path = ARTIFACTS / "bonus_prompts.json"

with eval_path.open("r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

# Выбираем два примера с изображениями и вопросами из реального набора
selected = []
for example in examples:
    if example.get("question") and example.get("image"):
        selected.append(example)
    if len(selected) == 2:
        break

if len(selected) < 2:
    raise ValueError("В eval.jsonl должно быть не менее двух примеров с изображениями и вопросами.")

results = []

for i, example in enumerate(selected, start=1):
    question = example["question"]

    # Альтернативная формулировка того же вопроса
    alternative_question = (
        f"Внимательно изучи изображение и ответь кратко и конкретно: {question}"
    )

    original_answer = run_vlm(
        example["image"], question, save=False
    )
    alternative_answer = run_vlm(
        example["image"], alternative_question, save=False
    )

    results.append({
        "example_id": example.get("id", i),
        "task": example.get("task"),
        "image": example["image"],
        "original_question": question,
        "alternative_question": alternative_question,
        "original_answer": original_answer["raw_response"],
        "alternative_answer": alternative_answer["raw_response"],
        "qualitative_comment": (
            "Сравните ответы: изменились ли содержание, конкретность, "
            "формат или полнота ответа. Комментарий следует заполнить "
            "после просмотра результатов."
        )
    })

output_path.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"Сохранено примеров: {len(results)}")
print(f"Файл: {output_path.resolve()}")

for result in results:
    print(f"\n--- Пример {result['example_id']} ---")
    print("Исходный вопрос:", result["original_question"])
    print("Исходный ответ:", result["original_answer"])
    print("Новый вопрос:", result["alternative_question"])
    print("Новый ответ:", result["alternative_answer"])

Сохранено примеров: 2
Файл: C:\Anton\16 триместр\SOTA\lab6\artifacts\bonus_prompts.json

--- Пример captioning:train:0 ---
Исходный вопрос: Describe the image.
Исходный ответ: This is a close-up photograph of a single bird perched on a light blue, weathered concrete ledge. The bird is the central focus, captured in profile as it looks towards the right of the frame.

- **Bird's Appearance**: The bird is predominantly black, with a glossy, iridescent sheen on its back and wings that reflects a deep, rich blue-purple. It has a long, slender, pointed beak that is dark and slightly curved. Its eyes are a striking, bright yellow, and its legs and feet are a dark greyish-black, with sharp, curved claws gripping the ledge.

- **Posture
Новый вопрос: Внимательно изучи изображение и ответь кратко и конкретно: Describe the image.
Новый ответ: A black bird with iridescent blue-green feathers on its back and wings, a long tail, and a sharp, pointed beak is perched on a light blue concrete ledge. T

## Анализ результатов

В ходе работы модель Qwen3-VL-2B была проверена на 20 реальных примерах: по 5 для captioning, VQA, OCR и structured extraction. Также был выполнен запуск на 160 ablation примерах для сравнения разных промптов, разрешения изображений и способов генерации. Модель работала на CPU в формате FP32 без квантования, поэтому основными затратами были время инференса и вычислительные ресурсы.

Результаты показали, что качество зависит от типа задачи и настроек. Для описания изображений лучше подходят подробные промпты, а для OCR — краткие и конкретные ответы. В задаче извлечения структурированных данных точность полей составила 90%. При этом в базовой оценке VQA и OCR точное совпадение ответов составило 0%, что говорит о проблемах с форматом ответов и их соответствием эталону. Среди ошибок чаще встречались пропущенная информация (3 случая), неправильное определение объекта (2 случая), ошибки в атрибутах и подсчёте объектов.

Есть и ограничения эксперимента: выборка небольшая, поэтому результаты нельзя считать универсальными. Кроме того, запуск на CPU оказался медленным, а модель не использовала NF4-квантование. В целом эксперимент показал, что выбор промпта заметно влияет на результат, а для более надёжных выводов нужно проверить модель на большем количестве данных.